# 🚀 SaaS Customer Churn Prediction
### TechFlow Project Management Software
---
**What this notebook does:**
1. Loads customer data and explores patterns (EDA)
2. Applies NLP sentiment analysis on support tickets
3. Trains machine learning models to predict churn
4. Triggers automated alerts for at-risk customers

**Dataset:** 2,500 customers | Train: 500 rows | Test: 2,000 rows


In [ ]:
# ============================================================
# CELL 1 - Install & Import all required libraries
# ============================================================
# If any library is missing, uncomment and run the line below:
# !pip install pandas matplotlib seaborn scikit-learn textblob requests

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from textblob import TextBlob
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Show plots inline inside the notebook
%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 110,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False
})

print("All libraries imported successfully!")


## 📂 Step 1 — Load the Data

In [ ]:
# ============================================================
# CELL 2 - Load train and test CSV files
# ============================================================
train = pd.read_csv('churn_dataset.csv')
test  = pd.read_csv('charn_dataset2.csv')

print("Train shape:", train.shape, "  <-- rows x columns")
print("Test shape :", test.shape)
print()
print("Churn value counts in training set:")
print(train['Churn'].value_counts())
print()
print("First 5 rows of training data:")
train.head()


In [ ]:
# ============================================================
# CELL 3 - Check data types and missing values
# ============================================================
print("Column data types:")
print(train.dtypes)
print()
print("Missing values in each column:")
print(train.isnull().sum())
print()
print("Basic statistics:")
train[['Account_Age_Days', 'Daily_Usage_Mins']].describe()


## 📊 Step 2 — Exploratory Data Analysis (EDA)

In [ ]:
# ============================================================
# CELL 4 - Plot 1: Churn Distribution
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.suptitle('How Many Customers Churned?', fontsize=14, fontweight='bold')

labels  = ['Retained (0)', 'Churned (1)']
sizes   = train['Churn'].value_counts().sort_index().values
colors  = ['#457B9D', '#E63946']

# Pie chart
axes[0].pie(sizes, labels=labels, autopct='%1.1f%%',
            colors=colors, explode=(0, 0.06), startangle=90,
            textprops={'fontsize': 11})
axes[0].set_title('Proportion', fontweight='bold')

# Bar chart
bars = axes[1].bar(labels, sizes, color=colors,
                   edgecolor='white', linewidth=1.5, width=0.45)
for b, v in zip(bars, sizes):
    axes[1].text(b.get_x() + b.get_width()/2, v + 3,
                 str(v), ha='center', fontweight='bold', fontsize=11)
axes[1].set_title('Count', fontweight='bold')
axes[1].set_ylabel('Number of Customers')
axes[1].set_ylim(0, 380)

plt.tight_layout()
plt.show()

total    = len(train)
churned  = train['Churn'].sum()
retained = total - churned
print(f"Churned  : {churned}  ({churned/total*100:.1f}%)")
print(f"Retained : {retained} ({retained/total*100:.1f}%)")
print(f"Naive baseline (always predict retained): {retained/total*100:.1f}%")
print("Our model must beat this baseline!")


In [ ]:
# ============================================================
# CELL 5 - Plot 2: Login Frequency vs Churn
# ============================================================
freq_churn = (train
              .groupby(['Login_Frequency', 'Churn'])
              .size()
              .unstack(fill_value=0))
freq_churn.columns = ['Retained', 'Churned']
freq_churn = freq_churn.loc[['Daily', 'Weekly', 'Rarely']]
churn_rate = (freq_churn['Churned'] / freq_churn.sum(axis=1) * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Login Frequency vs Churn', fontsize=14, fontweight='bold')

# Left - raw counts
freq_churn.plot(kind='bar', ax=axes[0],
                color=['#457B9D', '#E63946'],
                edgecolor='white', width=0.6)
axes[0].set_title('Raw Counts', fontweight='bold')
axes[0].set_xlabel('Login Frequency')
axes[0].set_ylabel('Number of Customers')
axes[0].set_xticklabels(freq_churn.index, rotation=0)

# Right - churn rate %
bar_colors = ['#457B9D', '#F4A261', '#E63946']
bars = axes[1].bar(churn_rate.index, churn_rate.values,
                   color=bar_colors, edgecolor='white', width=0.45)
for b, v in zip(bars, churn_rate.values):
    axes[1].text(b.get_x() + b.get_width()/2, v + 0.5,
                 f'{v}%', ha='center', fontweight='bold', fontsize=12)
avg = train['Churn'].mean() * 100
axes[1].axhline(y=avg, color='gray', linestyle='--',
                alpha=0.6, label=f'Avg: {avg:.1f}%')
axes[1].set_title('Churn Rate %', fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, 100)
axes[1].legend()

plt.tight_layout()
plt.show()

print("Key finding - Churn rate by login frequency:")
for freq, rate in churn_rate.items():
    bar = '█' * int(rate // 5)
    print(f"  {freq:8s} : {rate:5.1f}%  {bar}")
print()
print("INSIGHT: Customers who rarely log in churn at 80%+ rate!")


In [ ]:
# ============================================================
# CELL 6 - Plot 3: Usage Pattern Distributions
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Usage Patterns: Churned vs Retained Customers',
             fontsize=14, fontweight='bold')

features = [
    ('Account_Age_Days', axes[0], 'Account Age (days)'),
    ('Daily_Usage_Mins', axes[1], 'Daily Usage (minutes)'),
]

for feature, ax, title in features:
    for label, color in [('Retained', '#457B9D'), ('Churned', '#E63946')]:
        churn_val = 1 if label == 'Churned' else 0
        subset = train[train['Churn'] == churn_val][feature]
        ax.hist(subset, bins=25, alpha=0.6, color=color,
                label=label, edgecolor='white', linewidth=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(title)
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.show()

print("Mean comparison (Retained vs Churned):")
for col in ['Account_Age_Days', 'Daily_Usage_Mins']:
    r_mean = train[train['Churn'] == 0][col].mean()
    c_mean = train[train['Churn'] == 1][col].mean()
    print(f"  {col}:")
    print(f"    Retained avg : {r_mean:.1f}")
    print(f"    Churned avg  : {c_mean:.1f}")
    print()


## 🔤 Step 3 — NLP: Sentiment Analysis on Support Tickets

In [ ]:
# ============================================================
# CELL 7 - Compute sentiment scores using TextBlob
# TextBlob polarity: -1.0 = very negative, +1.0 = very positive
# ============================================================
print("Computing sentiment scores... (takes 10-20 seconds)")

for df in [train, test]:
    df['sentiment'] = df['Last_Support_Ticket'].apply(
        lambda text: TextBlob(str(text)).sentiment.polarity
    )

print("Done!")
print()

# Show example tickets with their scores
print("Example support tickets with sentiment scores:")
print("-" * 65)
sample = train[['Last_Support_Ticket', 'sentiment', 'Churn']].sample(6, random_state=42)
for _, row in sample.iterrows():
    status = "CHURNED" if row['Churn'] == 1 else "retained"
    score  = row['sentiment']
    emoji  = "😠" if score < -0.1 else "😊" if score > 0.1 else "😐"
    print(f"{emoji} [{score:+.2f}] ({status})")
    print(f"   {row['Last_Support_Ticket'][:70]}")
    print()


In [ ]:
# ============================================================
# CELL 8 - Plot 4: Sentiment Analysis Charts
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('NLP: Sentiment of Support Tickets',
             fontsize=14, fontweight='bold')

# Left - distribution
for label, color in [('Retained (0)', '#457B9D'), ('Churned (1)', '#E63946')]:
    churn_val = 1 if 'Churned' in label else 0
    subset = train[train['Churn'] == churn_val]['sentiment']
    axes[0].hist(subset, bins=20, alpha=0.6, color=color,
                 label=label, edgecolor='white')
axes[0].set_title('Sentiment Score Distribution', fontweight='bold')
axes[0].set_xlabel('Polarity  (-1 = negative  |  +1 = positive)')
axes[0].set_ylabel('Count')
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.5, label='Neutral (0)')
axes[0].legend()

# Right - average per group
avg_sent = train.groupby('Churn')['sentiment'].mean()
bars = axes[1].bar(['Retained (0)', 'Churned (1)'],
                   avg_sent.values,
                   color=['#457B9D', '#E63946'],
                   edgecolor='white', linewidth=1.5, width=0.4)
for b, v in zip(bars, avg_sent.values):
    axes[1].text(b.get_x() + b.get_width()/2, v + 0.003,
                 f'{v:.3f}', ha='center', fontweight='bold')
axes[1].set_title('Average Sentiment by Churn Status', fontweight='bold')
axes[1].set_ylabel('Mean Sentiment Polarity')
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f"Average sentiment - Retained customers : {avg_sent[0]:+.4f}")
print(f"Average sentiment - Churned customers  : {avg_sent[1]:+.4f}")
print()
print("INSIGHT: Churned customers write more negative support tickets!")


## 🤖 Step 4 — Machine Learning

In [ ]:
# ============================================================
# CELL 9 - Feature Engineering
# Convert text categories to numbers for the model
# ============================================================

# Encode Login_Frequency: Daily / Weekly / Rarely -> numbers
le = LabelEncoder()
train['login_freq_enc'] = le.fit_transform(train['Login_Frequency'])
test['login_freq_enc']  = le.transform(test['Login_Frequency'])

print("Login Frequency encoding:")
for cls, enc in zip(le.classes_, le.transform(le.classes_)):
    print(f"  {cls} -> {enc}")

# Final features we will use
FEATURES = ['Account_Age_Days', 'Daily_Usage_Mins', 'login_freq_enc', 'sentiment']

X_train = train[FEATURES]
y_train = train['Churn']
X_test  = test[FEATURES]
y_test  = test['Churn']

print()
print("Features used for modelling:", FEATURES)
print("Training samples :", len(X_train))
print("Test samples     :", len(X_test))


In [ ]:
# ============================================================
# CELL 10 - Train 3 models and compare accuracy
# ============================================================
models = {
    'Logistic Regression' : LogisticRegression(max_iter=500, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
print("Training and evaluating models on test set...")
print()
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc   = accuracy_score(y_test, preds)
    results[name] = {'acc': acc, 'model': model, 'preds': preds}
    print(f"  {name:25s} -> {acc*100:.1f}% accuracy")

best_name = max(results, key=lambda n: results[n]['acc'])
print()
print(f"Best model: {best_name} ({results[best_name]['acc']*100:.1f}%)")
print(f"Baseline (naive): 65.8%")
print(f"Improvement: +{results[best_name]['acc']*100 - 65.8:.1f} percentage points")


In [ ]:
# ============================================================
# CELL 11 - Plot 5: Model Comparison + Feature Importance
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')

model_names = list(results.keys())
accs        = [results[n]['acc'] * 100 for n in model_names]
bar_colors  = ['#1D3557', '#457B9D', '#A8DADC']

# Left - accuracy
bars = axes[0].barh(model_names, accs, color=bar_colors,
                    edgecolor='white', linewidth=1.5)
for b, v in zip(bars, accs):
    axes[0].text(v + 0.2, b.get_y() + b.get_height()/2,
                 f'{v:.1f}%', va='center', fontweight='bold')
axes[0].axvline(x=65.8, color='gray', linestyle='--',
                alpha=0.6, label='Naive baseline 65.8%')
axes[0].set_title('Test Set Accuracy', fontweight='bold')
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_xlim(0, 105)
axes[0].legend(fontsize=9)

# Right - feature importance from best model
best_model = results[best_name]['model']
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
else:
    importances = np.abs(best_model.coef_[0])
    importances = importances / importances.sum()

feat_labels = ['Account Age', 'Daily Usage', 'Login Freq', 'Sentiment']
sorted_idx  = np.argsort(importances)
axes[1].barh([feat_labels[i] for i in sorted_idx],
             importances[sorted_idx],
             color='#457B9D', edgecolor='white', linewidth=1.5)
axes[1].set_title(f'Feature Importance\n({best_name})', fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# CELL 12 - Plot 6: Confusion Matrix
# ============================================================
best_preds = results[best_name]['preds']
cm  = confusion_matrix(y_test, best_preds)
acc = results[best_name]['acc']

fig, ax = plt.subplots(figsize=(6, 5))
fig.suptitle(f'Confusion Matrix: {best_name}', fontsize=13, fontweight='bold')

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Predicted: Retained', 'Predicted: Churned'],
            yticklabels=['Actual: Retained',    'Actual: Churned'],
            linewidths=1, linecolor='white',
            cbar=False, annot_kws={'fontsize': 14})

ax.set_ylabel('Actual',    fontweight='bold')
ax.set_xlabel('Predicted', fontweight='bold')
ax.text(1, 2.3, f'Overall Accuracy: {acc*100:.1f}%',
        ha='center', fontsize=12, fontweight='bold', color='#1D3557')

plt.tight_layout()
plt.show()

print(f"True Negatives  (predicted retained, was retained) : {cm[0,0]}")
print(f"False Positives (predicted churned,  was retained) : {cm[0,1]}")
print(f"False Negatives (predicted retained, was churned)  : {cm[1,0]}  <- costly!")
print(f"True Positives  (predicted churned,  was churned)  : {cm[1,1]}")
print()
print("Full Classification Report:")
print(classification_report(y_test, best_preds, target_names=['Retained', 'Churned']))


## 🚨 Step 5 — Score Customers & Trigger Alerts

In [ ]:
# ============================================================
# CELL 13 - Score every customer with churn probability
# ============================================================
HIGH_RISK_THRESHOLD  = 0.75   # 75% and above = HIGH risk
MEDIUM_RISK_THRESHOLD = 0.50  # 50% and above = MEDIUM risk

scored = test.copy()
scored['churn_probability'] = results[best_name]['model'].predict_proba(X_test)[:, 1]
scored['churn_score_pct']   = (scored['churn_probability'] * 100).round(1).astype(str) + '%'
scored['risk_level'] = scored['churn_probability'].apply(
    lambda s: 'HIGH'   if s >= HIGH_RISK_THRESHOLD  else
              'MEDIUM' if s >= MEDIUM_RISK_THRESHOLD else 'LOW'
)

high   = scored[scored['risk_level'] == 'HIGH']
medium = scored[scored['risk_level'] == 'MEDIUM']
low    = scored[scored['risk_level'] == 'LOW']

print(f"HIGH risk customers   (>= {HIGH_RISK_THRESHOLD*100:.0f}%) : {len(high):,}")
print(f"MEDIUM risk customers (>= {MEDIUM_RISK_THRESHOLD*100:.0f}%) : {len(medium):,}")
print(f"LOW risk customers              : {len(low):,}")
print()
print("Top 10 highest risk customers:")
cols = ['Name', 'churn_score_pct', 'risk_level', 'Login_Frequency', 'sentiment']
scored.sort_values('churn_probability', ascending=False)[cols].head(10)


In [ ]:
# ============================================================
# CELL 14 - Simulate automated alert workflows
# Set email_enabled / slack_enabled to True when ready
# ============================================================
import datetime

EMAIL_ENABLED   = False   # set True + fill credentials to send real emails
SLACK_ENABLED   = False   # set True + fill webhook URL for real Slack alerts

def simulate_alert(row):
    risk  = row['risk_level']
    name  = row.get('Name', 'Unknown')
    score = row['churn_score_pct']
    freq  = row['Login_Frequency']
    sent  = round(row['sentiment'], 2)
    ticket = str(row.get('Last_Support_Ticket', ''))[:80]
    ts    = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')

    print(f"{'='*55}")
    print(f"  {'URGENT ' if risk=='HIGH' else ''}CHURN ALERT - {risk} RISK")
    print(f"{'='*55}")
    print(f"  Customer   : {name}")
    print(f"  Score      : {score}")
    print(f"  Login      : {freq}")
    print(f"  Sentiment  : {sent}")
    print(f"  Ticket     : {ticket}...")
    print(f"  Time       : {ts}")
    print()
    print(f"  Workflows fired:")
    if risk == 'HIGH':
        print(f"    [EMAIL]   -> CS team notified" if EMAIL_ENABLED else "    [EMAIL]   -> SIMULATED (set EMAIL_ENABLED=True)")
        print(f"    [SLACK]   -> #customer-success posted" if SLACK_ENABLED else "    [SLACK]   -> SIMULATED (set SLACK_ENABLED=True)")
        print(f"    [CRM]     -> Follow-up task created")
        print(f"    [ACTION]  -> Personal call within 24 hours")
    else:
        print(f"    [SLACK]   -> SIMULATED notification")
        print(f"    [ACTION]  -> Check-in email within 48 hours")
    print()

# Show alerts for top 5 high-risk customers
print("Showing alerts for top 5 HIGH risk customers:")
print()
top5 = scored[scored['risk_level'] == 'HIGH'].sort_values(
    'churn_probability', ascending=False).head(5)

for _, row in top5.iterrows():
    simulate_alert(row)


In [ ]:
# ============================================================
# CELL 15 - Save scored customers to CSV
# ============================================================
output_cols = ['Customer_ID', 'Name', 'Email', 'churn_score_pct',
               'risk_level', 'Login_Frequency', 'Account_Age_Days',
               'Daily_Usage_Mins', 'sentiment', 'Last_Support_Ticket']
output_cols = [c for c in output_cols if c in scored.columns]

scored[output_cols].sort_values(
    'churn_probability' if 'churn_probability' in scored.columns else output_cols[0],
    ascending=False
).to_csv('scored_customers.csv', index=False)

print("Saved -> scored_customers.csv")
print()
print("Summary:")
print(f"  Total customers scored : {len(scored):,}")
print(f"  HIGH risk              : {len(high):,}")
print(f"  MEDIUM risk            : {len(medium):,}")
print(f"  LOW risk               : {len(low):,}")
print(f"  Model accuracy         : {results[best_name]['acc']*100:.1f}%")
print()
print("Open scored_customers.csv in Excel to see all scores!")


## ✅ Project Complete!

### What we built:
| Step | What | Result |
|------|------|--------|
| EDA | Explored churn patterns | Rarely-login users churn at 82%+ |
| NLP | Sentiment on support tickets | Churned users have negative sentiment |
| ML  | Trained 3 models | 83.9% accuracy (Logistic Regression) |
| Alerts | Automated workflow triggers | 64 HIGH risk, 608 MEDIUM risk flagged |

### Key business insight:
> Customers who **rarely log in** AND write **negative support tickets** are nearly certain to churn. Target them with personal outreach **before** they cancel.


## 📧 Step 6 — Automated Retention Email (Sent Directly to Customer)

This is **different** from the CS team alert.  
This email goes **directly to the at-risk customer** with a personalised message and a discount code to win them back before they cancel.

| Setting | Value |
|---|---|
| Triggered for | HIGH + MEDIUM risk customers |
| Contains | Personalised message based on login frequency |
| Includes | Discount code to win them back |
| To activate | Set `EMAIL_ENABLED = True` and fill in Gmail credentials |


In [ ]:
# ============================================================
# CELL 16 - Configuration for Retention Email
# Set EMAIL_ENABLED = True and fill credentials to send real emails
# ============================================================
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import datetime

# ── Settings - edit these ─────────────────────────────────────────────────────
EMAIL_ENABLED   = False                        # change to True to send real emails
SENDER_EMAIL    = "your-gmail@gmail.com"       # your Gmail address
SENDER_PASSWORD = "your-app-password"          # Gmail App Password (not real password)
                                               # Get it at: myaccount.google.com/apppasswords
PRODUCT_NAME    = "TechFlow"
DISCOUNT_CODE   = "STAYWITHUS20"              # 20% off discount code
SUPPORT_URL     = "https://techflow.io/support"

print("Retention email settings loaded.")
print(f"  Email sending : {'ENABLED (real emails will be sent!)' if EMAIL_ENABLED else 'SIMULATED (no emails sent)'}")
print(f"  Discount code : {DISCOUNT_CODE}")
print(f"  Product name  : {PRODUCT_NAME}")


In [ ]:
# ============================================================
# CELL 17 - Build the personalised retention email body
# Message changes based on how often the customer logs in
# ============================================================

def build_retention_email(name, freq, email_addr):
    """
    Creates a personalised email body based on login frequency.
    - Rarely   -> 'we noticed you haven't logged in' message
    - Weekly   -> 'help you get more value' message
    - Daily    -> 'making sure you have everything you need' message
    """
    first_name = str(name).split()[0]   # use first name only

    if freq == "Rarely":
        personal_line = (
            f"We noticed you haven't been logging into {PRODUCT_NAME} much lately "
            f"and we'd love to know why. Has something made it harder to use?"
        )
        cta = "We'd love to show you some features you might have missed."

    elif freq == "Weekly":
        personal_line = (
            f"We can see you've been using {PRODUCT_NAME} occasionally "
            f"and we want to help you get even more value out of it."
        )
        cta = f"Our team has put together some tips to help you get the most out of {PRODUCT_NAME}."

    else:  # Daily
        personal_line = (
            f"You've been a valued {PRODUCT_NAME} user and we want to make sure "
            f"you're getting everything you need from us."
        )
        cta = f"We're always improving {PRODUCT_NAME} and we'd love your feedback."

    body = (
        f"Hi {first_name},\n\n"
        f"{personal_line}\n\n"
        f"{cta}\n\n"
        f"As a special thank-you, here is an exclusive offer just for you:\n\n"
        f"    Discount Code : {DISCOUNT_CODE}\n"
        f"    Savings       : 20% off your next 3 months\n\n"
        f"Apply this code at checkout, or simply reply to this email\n"
        f"and our team will apply it for you directly.\n\n"
        f"If something hasn't been working for you, we genuinely want to hear about it.\n"
        f"Visit our support centre: {SUPPORT_URL}\n\n"
        f"Warm regards,\n"
        f"The {PRODUCT_NAME} Customer Success Team\n"
        f"\n--\n"
        f"To unsubscribe, reply with 'unsubscribe' in the subject line."
    )
    return body.strip()


# Preview what the email looks like for 3 different customer types
print("=" * 60)
print("  RETENTION EMAIL PREVIEW")
print("=" * 60)

for freq_type in ["Rarely", "Weekly", "Daily"]:
    print(f"\n--- Preview for {freq_type} login user ---")
    sample = scored[scored["Login_Frequency"] == freq_type].iloc[0]
    preview = build_retention_email(sample["Name"], freq_type, sample.get("Email",""))
    print(preview[:500] + "...")
    print()


In [ ]:
# ============================================================
# CELL 18 - Send retention email function
# ============================================================

def send_retention_email(name, email_addr, freq, score_pct):
    """
    Sends (or simulates) a retention email to one customer.
    
    Parameters:
        name       - customer name
        email_addr - customer email address
        freq       - Login_Frequency (Daily/Weekly/Rarely)
        score_pct  - churn score percentage string e.g. '82.3%'
    """
    first_name = str(name).split()[0]
    subject    = f"We miss you, {first_name}! Here is 20% off from {PRODUCT_NAME}"
    body       = build_retention_email(name, freq, email_addr)

    if not EMAIL_ENABLED:
        # Simulation mode - just print what would be sent
        print(f"  [SIMULATED] -> {name:25s} ({score_pct}) at {email_addr}")
        return "simulated"

    # Real send via Gmail SMTP
    if not email_addr or "@" not in str(email_addr):
        print(f"  [SKIPPED]   -> {name} (no valid email address)")
        return "skipped"

    msg = MIMEMultipart("alternative")
    msg["Subject"] = subject
    msg["From"]    = SENDER_EMAIL
    msg["To"]      = email_addr
    msg.attach(MIMEText(body, "plain"))

    try:
        with smtplib.SMTP("smtp.gmail.com", 587) as server:
            server.starttls()
            server.login(SENDER_EMAIL, SENDER_PASSWORD)
            server.sendmail(SENDER_EMAIL, email_addr, msg.as_string())
        print(f"  [SENT]      -> {name} at {email_addr}")
        return "sent"
    except Exception as e:
        print(f"  [ERROR]     -> {name} : {e}")
        return "error"


print("send_retention_email() function ready!")
print()
print("It will SIMULATE emails until you set EMAIL_ENABLED = True in Cell 16.")


In [ ]:
# ============================================================
# CELL 19 - Run the retention email campaign
# Sends to all HIGH + MEDIUM risk customers
# ============================================================

# Get all at-risk customers sorted by churn probability
at_risk = scored[scored["risk_level"].isin(["HIGH", "MEDIUM"])].copy()
at_risk = at_risk.sort_values("churn_probability", ascending=False)

print(f"Starting retention email campaign...")
print(f"  Targeting: {len(at_risk):,} at-risk customers")
print(f"  HIGH risk  : {len(at_risk[at_risk['risk_level']=='HIGH']):,}")
print(f"  MEDIUM risk: {len(at_risk[at_risk['risk_level']=='MEDIUM']):,}")
print(f"  Mode       : {'LIVE - real emails sending!' if EMAIL_ENABLED else 'SIMULATION - no real emails'}")
print()
print("-" * 60)

# Track results
results_count = {"sent": 0, "simulated": 0, "skipped": 0, "error": 0}

# Only show first 10 in notebook output to avoid flooding
display_limit = 10
print(f"Showing first {display_limit} customers (all {len(at_risk):,} will be processed):")
print()

for i, (_, row) in enumerate(at_risk.iterrows()):
    name      = row.get("Name", "Unknown")
    email_addr = row.get("Email", "")
    freq      = row["Login_Frequency"]
    score_pct = row["churn_score_pct"]
    risk      = row["risk_level"]

    if i < display_limit:
        risk_label = "HIGH  " if risk == "HIGH" else "MEDIUM"
        print(f"  [{risk_label}] ", end="")

    status = send_retention_email(name, email_addr, freq, score_pct)
    results_count[status] = results_count.get(status, 0) + 1

    if i == display_limit - 1 and len(at_risk) > display_limit:
        print(f"  ... ({len(at_risk) - display_limit} more customers processed silently)")

print()
print("-" * 60)
print(f"Campaign complete!")
print(f"  Simulated : {results_count.get('simulated', 0):,}")
print(f"  Sent      : {results_count.get('sent', 0):,}")
print(f"  Skipped   : {results_count.get('skipped', 0):,}")
print(f"  Errors    : {results_count.get('error', 0):,}")
print()
if not EMAIL_ENABLED:
    print("To send REAL emails:")
    print("  1. Go to Cell 16")
    print("  2. Set EMAIL_ENABLED = True")
    print("  3. Fill in SENDER_EMAIL and SENDER_PASSWORD")
    print("  4. Re-run cells 16, 17, 18, 19")


In [ ]:
# ============================================================
# CELL 20 - Visual summary of the full campaign
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Retention Email Campaign Summary', fontsize=14, fontweight='bold')

# Left - risk breakdown pie
risk_counts = at_risk['risk_level'].value_counts()
colors_pie  = ['#E63946', '#F4A261']
axes[0].pie(risk_counts.values, labels=risk_counts.index,
            autopct='%1.1f%%', colors=colors_pie,
            explode=[0.05]*len(risk_counts),
            startangle=90, textprops={'fontsize': 11})
axes[0].set_title(f'Customers Targeted\n({len(at_risk):,} total)', fontweight='bold')

# Right - login frequency breakdown of at-risk customers
freq_counts = at_risk['Login_Frequency'].value_counts().loc[['Daily','Weekly','Rarely']]
bar_colors  = ['#457B9D', '#F4A261', '#E63946']
bars = axes[1].bar(freq_counts.index, freq_counts.values,
                   color=bar_colors, edgecolor='white', linewidth=1.5, width=0.5)
for b, v in zip(bars, freq_counts.values):
    axes[1].text(b.get_x() + b.get_width()/2, v + 1,
                 str(v), ha='center', fontweight='bold', fontsize=11)
axes[1].set_title('At-Risk Customers by Login Frequency', fontweight='bold')
axes[1].set_ylabel('Number of Customers')
axes[1].set_xlabel('Login Frequency')

plt.tight_layout()
plt.show()

print(f"Discount code used  : {DISCOUNT_CODE}")
print(f"Estimated open rate : ~25-30% (industry average for re-engagement emails)")
print(f"If 20% convert      : ~{int(len(at_risk)*0.20):,} customers saved")
avg_revenue = 50  # assumed monthly subscription
print(f"Estimated MRR saved : ~${int(len(at_risk)*0.20*avg_revenue):,}/month (at $50/customer)")


## ✅ Retention Email Campaign Complete!

### How the 4 automated workflows connect:

```
Customer churn score calculated
          |
          |-- score >= 75% (HIGH risk)
          |       |-- Email alert to CS team
          |       |-- Slack notification to #customer-success  
          |       |-- Retention email sent TO customer (with discount)
          |       |-- HubSpot CRM task created for account manager
          |
          |-- score >= 50% (MEDIUM risk)
                  |-- Slack notification
                  |-- Retention email sent TO customer (with discount)
```

### To activate real emails:
1. Go to **Cell 16** and set `EMAIL_ENABLED = True`
2. Fill in your Gmail address and [App Password](https://myaccount.google.com/apppasswords)
3. Re-run cells 16 → 19

### Gmail App Password (not your real password):
- Go to myaccount.google.com → Security → 2-Step Verification → App Passwords
- Create one for "Mail" → copy the 16-character code → paste as `SENDER_PASSWORD`
